# 20 v2 - Selective-refinement LIBERO-PRO worker: model 1

This backfills the established K=5, Euler-steps `(3,4)`, refine-last arm for the exact 130
identities already collected by the v2 diversity-signal workers. It requests both baseline and
refinement under the existing `pi05-diversity-signal-v2-m1` experiment. Supabase
rollout IDs are behavior-derived, so completed uncertainty-only rows are skipped and only missing
rows execute.

The model repository and immutable revision are read from the existing baseline run metadata.
This fails rather than silently evaluating a newer Hub upload. Run `SHARD_INDEX=0,1,2,3`; each
completed shard is resumable. Every ten new rollouts, the worker prints the stored baseline SR,
current refinement SR, and historical reference SR by suite.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (DIVERSITY_V2_EXPERIMENT_PREFIX,
    load_bootstrap_manifest, run_diversity_refinement_worker)

drive.mount("/content/drive")

MEMBER_INDEX = 1
EXPECTED_EPISODES_PER_SUITE = 10  # exact 130 stored baseline identities/member
SHARD_COUNT = 4
SHARD_INDEX = 0                # run 0, 1, 2, 3 for this member
EXPERIMENT_PREFIX = DIVERSITY_V2_EXPERIMENT_PREFIX
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json")
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest["source_model"] == PI05_REPO_ID, manifest["source_model"]

print({"member": MEMBER_INDEX, "experiment_prefix": EXPERIMENT_PREFIX,
       "expected_episodes_per_suite": EXPECTED_EPISODES_PER_SUITE,
       "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
       "manifest_hash": manifest["manifest_hash"]})
run_diversity_refinement_worker(
    member_index=MEMBER_INDEX,
    expected_episodes_per_suite=EXPECTED_EPISODES_PER_SUITE,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest["manifest_hash"],
    experiment_prefix=EXPERIMENT_PREFIX)